<a href="https://colab.research.google.com/github/riteshkrkumarweb/Learning_AI/blob/main/PCA_with_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
kreeshrajani_breast_cancer_survival_dataset_path = kagglehub.dataset_download('kreeshrajani/breast-cancer-survival-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kreeshrajani/breast-cancer-survival-dataset/breast_cancer_survival.csv


In [ ]:
df = pd.read_csv('/kaggle/input/datasets/kreeshrajani/breast-cancer-survival-dataset/breast_cancer_survival.csv',usecols=['Age',	'Gender',	'Protein1',	'Protein2'	,'Protein3'	,'Protein4','Patient_Status'])

In [ ]:
df

,Age,Gender,Protein1,Protein2,Protein3,Protein4,Patient_Status
0,42,FEMALE,0.952560,2.15000,0.007972,-0.048340,Alive
1,54,FEMALE,0.000000,1.38020,-0.498030,-0.507320,Dead
2,63,FEMALE,-0.523030,1.76400,-0.370190,0.010815,Alive
3,78,FEMALE,-0.876180,0.12943,-0.370380,0.132190,Alive
4,42,FEMALE,0.226110,1.74910,-0.543970,-0.390210,Alive
...,...,...,...,...,...,...,...
329,59,FEMALE,0.024598,1.40050,0.024751,0.280320,Alive
330,41,FEMALE,0.100120,-0.46547,0.472370,-0.523870,Alive
331,54,FEMALE,0.753820,1.64250,-0.332850,0.857860,Dead
332,74,FEMALE,0.972510,1.42680,-0.366570,-0.107820,Alive


In [ ]:
#Here we Change the status to the Alive to 1 and dead to 0
df['Patient_Status'] = df['Patient_Status'].map({'Alive': 1, 'Dead': 0}).astype('Int64')
#converting the female to 1 and male 2
df['Gender'] = df['Gender'].map({'FEMALE': 1, 'MALE': 2})

In [ ]:
df.sample(5)

,Age,Gender,Protein1,Protein2,Protein3,Protein4,Patient_Status
298,64,1,-0.24408,1.69070,-0.12854,-1.34020,1
80,57,1,0.12704,-0.49761,0.97681,0.62447,1
293,53,1,-0.36165,2.65660,0.30769,0.20382,1
16,45,1,-0.52601,0.71836,-0.67598,-0.83526,1
155,37,1,0.29490,1.36250,0.69238,0.41002,1


In [ ]:
#Here we have the Missing value so we have to Impute this
df.isnull().sum()

Age                0
Gender             0
Protein1           0
Protein2           0
Protein3           0
Protein4           0
Patient_Status    13
dtype: int64

In [ ]:
#With the help of the pandas we impute the values we cannot impute using the means
#beacuse the mean has flot value and we transform it to int64 above so we use median
df['Patient_Status'] = df['Patient_Status'].fillna(df['Patient_Status'].mean())

TypeError: Invalid value '0.794392523364486' for dtype 'Int64'

In [ ]:
#Here we use median and the data is imputed successfully
df['Patient_Status'] = df['Patient_Status'].fillna(df['Patient_Status'].median())
# Remove a single column beacuse all are the womens and four men but i not remove
#df.drop(columns='Gender', inplace=True)


In [ ]:
df.isnull().sum()

Age               0
Gender            0
Protein1          0
Protein2          0
Protein3          0
Protein4          0
Patient_Status    0
dtype: int64

In [ ]:
#Befor PCA

In [ ]:
#we devide the data first X and Y
# donot forget between -1 and 1
#X meaning training colum and Y means testing or Target Column
X = df.iloc[:,:-1] # Full row but not the last one it is target column donot forget between -1 and 1
y = df.iloc[:,-1]#Full row and only the last column it is the target column



In [ ]:
from sklearn.model_selection import train_test_split

# Split features (X) and target (y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train

,Age,Gender,Protein1,Protein2,Protein3,Protein4
224,38,1,-0.268450,0.19515,-1.024700,0.101720
78,77,1,-0.298700,-0.16129,0.460720,-0.396660
295,47,1,-0.190060,1.97790,-0.007615,0.035325
17,63,1,0.052728,0.72210,-0.308650,-0.531290
24,70,1,0.700290,0.97347,-0.296450,0.105510
...,...,...,...,...,...,...
188,44,1,-0.278840,2.16880,-0.462330,0.272200
71,45,1,-0.711630,1.69240,-0.800760,0.794400
106,49,1,0.061643,1.31490,-0.099357,0.754410
270,79,1,-0.482690,-0.31677,0.471580,0.347440


In [ ]:
'''
Standardizes features by removing the mean and scaling to unit variance,
transforming data to have mean = 0 and standard deviation = 1.'''
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit and transform training data
X_test_scaled = scaler.transform(X_test)  # Use same scaler on test data

In [ ]:
X_test_scaled

array([[ 1.34038864, -0.12332528,  0.123593  , -1.49361609,  0.94241979,
         1.68982932],
       [-0.50358096, -0.12332528, -0.29759387,  0.67299665, -0.13946226,
         1.34265311],
       [ 0.57206797, -0.12332528, -2.49887193,  0.22326746, -0.25318369,
        -0.36786365],
       [ 1.34038864, -0.12332528, -0.42198484, -0.35354642,  0.35794451,
         0.63232526],
       [-0.27308476, -0.12332528, -1.21247797,  1.1326026 , -1.19087892,
        -0.4221834 ],
       [ 0.95622831, -0.12332528,  0.83243258, -1.32623872, -0.27319056,
        -0.33195109],
       [ 0.95622831, -0.12332528, -0.38023771, -0.47490708, -0.45940039,
        -1.34276882],
       [-0.1962527 , -0.12332528,  0.4599967 ,  1.09736053,  1.03446855,
        -0.50001542],
       [-0.58041303, -0.12332528, -1.56862321,  0.83832011,  3.06340525,
        -0.12367113],
       [-0.8877413 , -0.12332528,  1.13964919,  0.01586677, -0.24902776,
         0.59322405],
       [-0.6572451 , -0.12332528, -0.25809567,  1.

In [ ]:
"""
After the Standard Scaling
What is KNN ?
   A lazy learning algorithm that predicts by finding the K most
   similar data points and using majority vote (classification)
   or average (regression) of their values.
"""
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier()

In [ ]:
#Train the data
knn.fit(X_train_scaled, y_train)

KNeighborsClassifier()

In [ ]:
#Make prediction
# Step 6: Make predictions
y_pred = knn.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")  # Better formatting

Accuracy: 0.79


In [ ]:
#After the PCA implitation
#Apply PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2)  # Reduce From 7 to 2 components
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
#The column are reduced to 2 from the 7 two most importand colms
print(f"Original training shape: {X_train_scaled.shape}")
print(f"PCA training shape: {X_train_pca.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")
print(f"Accuracy Score Before PCA:{accuracy * 100:.2f}%")#Means 2f only two float number

Original training shape: (267, 6)
PCA training shape: (267, 2)
Explained variance ratio: [0.26526544 0.20595356]
Total variance explained: 47.12%
Accuracy Score Before PCA:80.60%


In [ ]:
X_train_pca.shape


(267, 2)

In [ ]:
knn = KNeighborsClassifier()

In [ ]:
knn.fit(X_train_pca , y_train)

KNeighborsClassifier()

In [ ]:
y_predict = knn.predict(X_test_pca)

In [ ]:
accuracy = accuracy_score(y_test, y_predict)
print(f"Accuracy with PCA: {accuracy:.2f}")

Accuracy with PCA: 0.79


In [ ]:
#creating the loop that what number of components is important for PCA
#Like 1colum 2 colum or 4 colum n for best accr
for i in range(1,7):
    pca = PCA(n_components=i)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    knn = KNeighborsClassifier()
    knn.fit(X_train_pca , y_train)
    y_predict = knn.predict(X_test_pca)
    accuracy = accuracy_score(y_test, y_predict)
    print(f"Components: {i} PCA: {accuracy:.2f} ")




Components: 1 PCA: 0.78 
Components: 2 PCA: 0.79 
Components: 3 PCA: 0.82 
Components: 4 PCA: 0.81 
Components: 5 PCA: 0.82 
Components: 6 PCA: 0.79 
